# PACE Level 2
<br>

**Steps:**

* Create a plan for files to use `pc.plan()`
* Print the plan to check it `print(plan.summary())`
* Do the plan and get matchups `pc.matchup(plan)`

*Note: In a virtual machine in AWS us-west-2, where NASA cloud data is, the point matchups are fast. In Colab, say, your compute is not in the same data region nor provider, and the same matchups might take 10x longer.*

## Prerequisites

The examples here use NASA EarthData and you need to have an account with EarthData. Make sure you can login.

In [ ]:
# if needed
pip install point-collocation

In [1]:
import earthaccess
earthaccess.login()

## Here are the level 2 datasets

In [5]:
import earthaccess
results = earthaccess.search_datasets(instrument="oci")

short_names = [
    item.summary()["short-name"]
    for item in results
    if "L2" in item.summary()["short-name"]
]

print(short_names)

['PACE_OCI_L2_UVAI_UAA_NRT', 'PACE_OCI_L2_UVAI_UAA', 'PACE_OCI_L2_AER_UAA_NRT', 'PACE_OCI_L2_AER_UAA', 'PACE_OCI_L2_AOP_NRT', 'PACE_OCI_L2_AOP', 'PACE_OCI_L2_CLOUD_MASK_NRT', 'PACE_OCI_L2_CLOUD_MASK', 'PACE_OCI_L2_CLOUD_NRT', 'PACE_OCI_L2_CLOUD', 'PACE_OCI_L2_LANDVI_NRT', 'PACE_OCI_L2_LANDVI', 'PACE_OCI_L2_BGC_NRT', 'PACE_OCI_L2_BGC', 'PACE_OCI_L2_IOP_NRT', 'PACE_OCI_L2_IOP', 'PACE_OCI_L2_PAR_NRT', 'PACE_OCI_L2_PAR', 'PACE_OCI_L2_SFREFL_NRT', 'PACE_OCI_L2_SFREFL', 'PACE_OCI_L2_TRGAS_NRT', 'PACE_OCI_L2_TRGAS']


## Load some points

In [1]:
import pandas as pd
url = (
    "https://raw.githubusercontent.com/"
    "fish-pace/point-collocation/main/"
    "examples/fixtures/points.csv"
)
df_points = pd.read_csv(url)
print(len(df_points))
df_points.head()

595


,lat,lon,date
0,27.3835,-82.7375,2024-06-13
1,27.1190,-82.7125,2024-06-14
2,26.9435,-82.8170,2024-06-14
3,26.6875,-82.8065,2024-06-14
4,26.6675,-82.6455,2024-06-14


## Get a plan for matchups for 1st 50 points from PACE data

A time buffer is needed since the swath granules are short time windows. We set time_buffer="12h" to find granules that are within 12 hours of our point times.

In [2]:
%%time
import point_collocation as pc
plan = pc.plan(
    df_points[0:50], 	
    data_source="earthaccess",
    source_kwargs={
        "short_name": "PACE_OCI_L2_AOP",
    },
    time_buffer="12h"
)

CPU times: user 675 ms, sys: 67.2 ms, total: 742 ms
Wall time: 3.72 s


In [3]:
plan.summary()

Plan: 50 points → 13 unique granule(s)
  Points with 0 matches : 0
  Points with >1 matches: 10
  Time buffer: 0 days 12:00:00

First 5 point(s):
  [0] lat=27.3835, lon=-82.7375, time=2024-06-13 12:00:00: 2 match(es)
    → https://obdaac-tea.earthdatacloud.nasa.gov/ob-cumulus-prod-public/PACE_OCI.20240613T171620.L2.OC_AOP.V3_1.nc
    → https://obdaac-tea.earthdatacloud.nasa.gov/ob-cumulus-prod-public/PACE_OCI.20240613T184939.L2.OC_AOP.V3_1.nc
  [1] lat=27.1190, lon=-82.7125, time=2024-06-14 12:00:00: 1 match(es)
    → https://obdaac-tea.earthdatacloud.nasa.gov/ob-cumulus-prod-public/PACE_OCI.20240614T175104.L2.OC_AOP.V3_1.nc
  [2] lat=26.9435, lon=-82.8170, time=2024-06-14 12:00:00: 1 match(es)
    → https://obdaac-tea.earthdatacloud.nasa.gov/ob-cumulus-prod-public/PACE_OCI.20240614T175104.L2.OC_AOP.V3_1.nc
  [3] lat=26.6875, lon=-82.8065, time=2024-06-14 12:00:00: 1 match(es)
    → https://obdaac-tea.earthdatacloud.nasa.gov/ob-cumulus-prod-public/PACE_OCI.20240614T175104.L2.OC_AOP.V3_

## Look at the granule to see groups

We will use `plan.open_dataset(0)` to open the first granule and take a look. This uses open_method="auto". It will try `xr.open_dataset`, discover no lat/lon and then try `xr.open_datatree` + merge. If you know, the netcdfs are grouped, you can pass in `open_method="datatree-merge"` yourself. We want these two groups: `/geophysical_data` and `/navigation_data`. We will create a open method.

In [4]:
%%time
plan.open_dataset(0, open_method="datatree")

open_method: {'xarray_open': 'datatree', 'open_kwargs': {'chunks': {}, 'engine': 'h5netcdf', 'decode_timedelta': False}, 'merge': None, 'coords': 'auto', 'set_coords': True, 'dim_renames': None, 'auto_align_phony_dims': None}
CPU times: user 286 ms, sys: 46.1 ms, total: 332 ms
Wall time: 1.33 s


<xarray.DataTree>
Group: /
│   Attributes: (12/47)
│       title:                             OCI Level-2 Data AOP
│       product_name:                      PACE_OCI.20240613T171120.L2.OC_AOP.V3_...
│       processing_version:                3.1
│       history:                           l2gen par=/data18/sdpsoper/vdc/vpu37/w...
│       instrument:                        OCI
│       platform:                          PACE
│       ...                                ...
│       geospatial_lon_min:                -82.05421
│       startDirection:                    Ascending
│       endDirection:                      Ascending
│       day_night_flag:                    Day
│       earth_sun_distance_correction:     0.9694168567657471
│       geospatial_bounds:                 POLYGON ((-55.18162 31.32767, -82.0542...
├── Group: /sensor_band_parameters
│       Dimensions:        (number_of_bands: 286, number_of_reflective_bands: 286,
│                           wavelength_3d: 172)
│       Coordinates:
│         * wavelength_3d  (wavelength_3d) float64 1kB 346.0 348.0 351.0 ... 717.0 719.0
│       Dimensions without coordinates: number_of_bands, number_of_reflective_bands
│       Data variables:
│           wavelength     (number_of_bands) float64 2kB dask.array<chunksize=(32,), meta=np.ndarray>
│           vcal_gain      (number_of_reflective_bands) float32 1kB dask.array<chunksize=(32,), meta=np.ndarray>
│           vcal_offset    (number_of_reflective_bands) float32 1kB dask.array<chunksize=(32,), meta=np.ndarray>
│           F0             (number_of_reflective_bands) float32 1kB dask.array<chunksize=(32,), meta=np.ndarray>
│           aw             (number_of_reflective_bands) float32 1kB dask.array<chunksize=(32,), meta=np.ndarray>
│           bbw            (number_of_reflective_bands) float32 1kB dask.array<chunksize=(32,), meta=np.ndarray>
│           k_oz           (number_of_reflective_bands) float32 1kB dask.array<chunksize=(32,), meta=np.ndarray>
│           k_no2          (number_of_reflective_bands) float32 1kB dask.array<chunksize=(32,), meta=np.ndarray>
│           Tau_r          (number_of_reflective_bands) float32 1kB dask.array<chunksize=(32,), meta=np.ndarray>
├── Group: /scan_line_attributes
│       Dimensions:  (number_of_lines: 1710)
│       Dimensions without coordinates: number_of_lines
│       Data variables: (12/13)
│           year     (number_of_lines) float64 14kB dask.array<chunksize=(32,), meta=np.ndarray>
│           day      (number_of_lines) float64 14kB dask.array<chunksize=(32,), meta=np.ndarray>
│           msec     (number_of_lines) float64 14kB dask.array<chunksize=(32,), meta=np.ndarray>
│           time     (number_of_lines) datetime64[ns] 14kB dask.array<chunksize=(32,), meta=np.ndarray>
│           detnum   (number_of_lines) float32 7kB dask.array<chunksize=(32,), meta=np.ndarray>
│           mside    (number_of_lines) float32 7kB dask.array<chunksize=(32,), meta=np.ndarray>
│           ...       ...
│           clon     (number_of_lines) float32 7kB dask.array<chunksize=(32,), meta=np.ndarray>
│           elon     (number_of_lines) float32 7kB dask.array<chunksize=(32,), meta=np.ndarray>
│           slat     (number_of_lines) float32 7kB dask.array<chunksize=(32,), meta=np.ndarray>
│           clat     (number_of_lines) float32 7kB dask.array<chunksize=(32,), meta=np.ndarray>
│           elat     (number_of_lines) float32 7kB dask.array<chunksize=(32,), meta=np.ndarray>
│           csol_z   (number_of_lines) float32 7kB dask.array<chunksize=(32,), meta=np.ndarray>
├── Group: /geophysical_data
│       Dimensions:   (number_of_lines: 1710, pixels_per_line: 1272, wavelength_3d: 172)
│       Dimensions without coordinates: number_of_lines, pixels_per_line, wavelength_3d
│       Data variables:
│           Rrs       (number_of_lines, pixels_per_line, wavelength_3d) float32 1GB dask.array<chunksize=(32, 256, 40), meta=np.ndarray>
│           Rrs_unc   (number_of_lines, pixels_per_line, wave

In [5]:
pace_l2 = {'xarray_open': 'dataset', 'merge': ['/geophysical_data', '/navigation_data']}
plan.open_dataset(0, open_method=pace_l2)

open_method: {'xarray_open': 'dataset', 'merge': ['/geophysical_data', '/navigation_data'], 'open_kwargs': {'chunks': {}, 'engine': 'h5netcdf', 'decode_timedelta': False}, 'coords': 'auto', 'set_coords': True, 'dim_renames': None, 'auto_align_phony_dims': None, 'merge_kwargs': {}}


<xarray.Dataset> Size: 3GB
Dimensions:    (number_of_lines: 1710, pixels_per_line: 1272, wavelength_3d: 172)
Coordinates:
    longitude  (number_of_lines, pixels_per_line) float32 9MB dask.array<chunksize=(256, 1272), meta=np.ndarray>
    latitude   (number_of_lines, pixels_per_line) float32 9MB dask.array<chunksize=(256, 1272), meta=np.ndarray>
Dimensions without coordinates: number_of_lines, pixels_per_line, wavelength_3d
Data variables:
    Rrs        (number_of_lines, pixels_per_line, wavelength_3d) float32 1GB dask.array<chunksize=(32, 256, 40), meta=np.ndarray>
    Rrs_unc    (number_of_lines, pixels_per_line, wavelength_3d) float32 1GB dask.array<chunksize=(32, 256, 40), meta=np.ndarray>
    aot_865    (number_of_lines, pixels_per_line) float32 9MB dask.array<chunksize=(256, 1272), meta=np.ndarray>
    angstrom   (number_of_lines, pixels_per_line) float32 9MB dask.array<chunksize=(256, 1272), meta=np.ndarray>
    avw        (number_of_lines, pixels_per_line) float32 9MB dask.array<chunksize=(256, 1272), meta=np.ndarray>
    nflh       (number_of_lines, pixels_per_line) float32 9MB dask.array<chunksize=(256, 1272), meta=np.ndarray>
    l2_flags   (number_of_lines, pixels_per_line) int32 9MB dask.array<chunksize=(256, 1272), meta=np.ndarray>
    tilt       (number_of_lines) float32 7kB dask.array<chunksize=(32,), meta=np.ndarray>

## Get the matchups using that plan

`pc.matchup()` will open each L2 granule as a DataTree and merges all groups into a flat dataset. The spatial method will automatically use "ndpoint" for non-gridded data. You can also try `spatial_method="xoak"`, which uses a similar algorithm. 

Notice, that point 0 is matched to 2 granules and so has 2 rows with the same `pc_id`. Many matchups are NaN because there is no data (cloudy) for that lat/lon point.

In [6]:
%%time
# spatial_method="auto" defaults to "ndpoint" in this case
res = pc.matchup(plan, variables=["Rrs"], open_method=pace_l2)

CPU times: user 22.4 s, sys: 2.21 s, total: 24.7 s
Wall time: 43.9 s


In [7]:
res.head()

,lat,lon,time,pc_id,granule_id,granule_time,granule_lat,granule_lon,Rrs_0,Rrs_1,...,Rrs_162,Rrs_163,Rrs_164,Rrs_165,Rrs_166,Rrs_167,Rrs_168,Rrs_169,Rrs_170,Rrs_171
0,27.3835,-82.7375,2024-06-13 12:00:00,0,https://obdaac-tea.earthdatacloud.nasa.gov/ob-...,2024-06-13 17:18:49+00:00,27.443144,-82.612923,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,27.3835,-82.7375,2024-06-13 12:00:00,0,https://obdaac-tea.earthdatacloud.nasa.gov/ob-...,2024-06-13 18:52:08+00:00,27.383293,-82.721527,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,27.1190,-82.7125,2024-06-14 12:00:00,1,https://obdaac-tea.earthdatacloud.nasa.gov/ob-...,2024-06-14 17:53:34+00:00,27.101389,-82.717186,0.01299,0.012946,...,0.000238,0.000228,0.000198,0.000194,0.000186,0.000172,0.000152,0.000122,0.000108,0.000094
3,26.9435,-82.8170,2024-06-14 12:00:00,2,https://obdaac-tea.earthdatacloud.nasa.gov/ob-...,2024-06-14 17:53:34+00:00,26.954554,-82.810219,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,26.6875,-82.8065,2024-06-14 12:00:00,3,https://obdaac-tea.earthdatacloud.nasa.gov/ob-...,2024-06-14 17:53:34+00:00,26.703817,-82.817726,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
%%time
# we can specify to use the xoak package
res2 = pc.matchup(plan, 
                  spatial_method="xoak",
                  variables=["Rrs"], 
                  open_method=pace_l2)

CPU times: user 26.2 s, sys: 1.22 s, total: 27.4 s
Wall time: 42.7 s
